# 04 — Feature Engineering

## Objetivo
Transformar los 24 datos raw en un conjunto rico de features que capturen los patrones descubiertos en el notebook anterior. La idea: darle al modelo suficiente contexto para distinguir lo **normal** de lo **anómalo**.

## Estrategia de features

| Categoría | Qué captura | Ejemplos |
|-----------|-------------|----------|
| **Temporales** | Estacionalidad, ciclos | mes, trimestre, estación seca/lluviosa |
| **Rolling stats** | Comportamiento reciente vs histórico | media 7d, 30d, desviación estándar |
| **Lags** | Inercia temporal | valor del mes anterior, hace 1 año |
| **Ratios** | Relaciones entre variables | hidro/total, fósil/total |
| **Descomposición** | Tendencia vs anomalía | trend, residual, z-score |

In [ ]:
import sys, warnings
sys.path.insert(0, '../..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv('../../data/raw/ecuador_electricity_real.csv', parse_dates=['fecha'])
df.columns = [c.replace(',', '_').replace(' ', '_') for c in df.columns]
print(f"Datos originales: {df.shape[0]} filas x {df.shape[1]} columnas")

## 4.1 Features temporales

Capturamos ciclos estacionales y el contexto de calendario ecuatoriano.

In [ ]:
from src.processing.features import add_temporal_features

df = add_temporal_features(df, date_col='fecha')
temporal_cols = ['mes', 'trimestre', 'dia_anio', 'es_estacion_lluviosa']
print(f"Features temporales añadidas: {temporal_cols}")
df[['fecha'] + temporal_cols].head(12)

## 4.2 Rolling statistics

Las medias y desviaciones móviles capturan si el mes actual es **inusual** respecto al pasado reciente.

**¿Por qué ventanas de 7, 14 y 30?**
- **7 meses** ≈ semestre → variabilidad a corto plazo
- **14 meses** ≈ año y algo → captura estacionalidad completa
- **30 meses** ≈ 2.5 años → tendencia a mediano plazo

In [ ]:
from src.processing.features import engineer_features

# Aplicar todo el pipeline de features
value_cols = ['gen_bioenergy', 'gen_gas', 'gen_hydro', 'gen_other_fossil', 'gen_solar',
              'gen_total_generation', 'gen_wind', 'demanda_twh', 'co2_intensity_gco2_kwh',
              'importaciones_netas_twh']
value_cols = [c for c in value_cols if c in df.columns]

df_feat = engineer_features(df.copy(), date_col='fecha', value_cols=value_cols)

print(f"Antes: {df.shape[1]} columnas")
print(f"Después: {df_feat.shape[1]} columnas")
print(f"Features nuevos: {df_feat.shape[1] - df.shape[1]}")

# Mostrar tipos de features generados
new_cols = [c for c in df_feat.columns if c not in df.columns]
tipos = {}
for c in new_cols:
    if '_media_' in c: tipos.setdefault('Rolling media', []).append(c)
    elif '_std_' in c: tipos.setdefault('Rolling std', []).append(c)
    elif '_min_' in c or '_max_' in c: tipos.setdefault('Rolling min/max', []).append(c)
    elif '_lag_' in c: tipos.setdefault('Lags', []).append(c)
    elif '_zscore_' in c: tipos.setdefault('Z-scores', []).append(c)
    elif '_pct_change' in c: tipos.setdefault('Cambio %', []).append(c)
    elif '_trend' in c or '_residual' in c: tipos.setdefault('Descomposición', []).append(c)
    elif 'ratio' in c: tipos.setdefault('Ratios', []).append(c)
    else: tipos.setdefault('Temporales/Otros', []).append(c)

print("\nDesglose de features generados:")
for tipo, cols in sorted(tipos.items()):
    print(f"  {tipo}: {len(cols)}")

## 4.3 Ejemplo: ¿Cómo se ve un feature útil?

El **z-score de 30 meses** de la demanda nos dice cuántas desviaciones estándar se aleja el mes actual de su media reciente. Valores > 2 o < -2 son inusuales.

In [ ]:
import plotly.graph_objects as go

if 'demanda_twh_zscore_30d' in df_feat.columns:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_feat['fecha'], y=df_feat['demanda_twh_zscore_30d'],
                             mode='lines+markers', line=dict(color='#1976D2'),
                             marker=dict(size=5)))
    fig.add_hline(y=2, line_dash='dash', line_color='red', annotation_text='Umbral +2σ')
    fig.add_hline(y=-2, line_dash='dash', line_color='red', annotation_text='Umbral -2σ')
    fig.add_hline(y=0, line_color='gray', line_width=0.5)
    fig.add_vrect(x0='2024-09-15', x1='2024-12-31', fillcolor='rgba(255,0,0,0.1)')
    fig.update_layout(title='Z-Score de Demanda (ventana 30 meses) — Detecta desviaciones inusuales',
                      yaxis_title='Z-Score (σ)', template='plotly_white')
    fig.show()

## 4.4 Resumen

De **24 columnas originales** generamos **~213 features** que capturan:
- Contexto temporal y estacional de Ecuador
- Variabilidad reciente vs histórica (rolling)
- Inercia temporal (lags)
- Relaciones entre fuentes de generación (ratios)
- Desviaciones de la tendencia (z-scores, residuales)

Estos features le dan al modelo Isolation Forest la capacidad de evaluar cada mes desde **múltiples perspectivas**, no solo mirando un valor individual.

---
*Siguiente notebook: 05 — Selección de Modelo*